In [1]:
import requests
from bs4 import BeautifulSoup

Documentation: 
- https://beautiful-soup-4.readthedocs.io/en/latest/
- https://requests.readthedocs.io/en/latest/

# OG Webscraping: links and text

* If we don't have time (or energy) to simmulate a browser in order to do web scaping, we can do the same using links and text.
* Links are used to locate a specific web page (just a url)
* Text describes a web page and its components.
* You can fetch the text of a web page using its link through the `requests` library.
* You can organize and navigate through this text using the `bs4` library, short for Beautiful Soup 4.
* Most of web scraping can be done by creating rules to get links from a text and then doing the same for the text from those links.

## The Good
* Easier than having to deal with a driver.
* Less compute and memory intense.
* Allows for asynchronous scraping.
* Simpler toolset has an easier learning curve.

## The Bad
* Most modern websites will not return such a "nice" HTML text file.
* Javascript and reactive website features lead to the need of complicated interactions.
* "Lazy" content cannot be scraped this way with ease, and might otherwise need an driver.
    * Content that shows only if the user scrolls all the way down.
    * Content that shows only when the user hovers the mouse over a link.

## The Ugly
* Requires you to do a lot of "hard coded" and ad-hoc rules.
* No guidance of a simmulated driver.
* Interface is cruder and code is less intuitive.

In [70]:
root = "https://ca.finance.yahoo.com/topic/news/"
req = requests.get(root)
req

<Response [200]>

In [71]:
soup = BeautifulSoup(req.text, 'html.parser')

In [72]:
soup


<!DOCTYPE html>

<html class="desktop neo-green dock-upscale failsafe" data-color-scheme="light" lang="en-CA" theme="light">
<head>
<meta charset="utf-8"/>
<meta content="guce.yahoo.com" name="oath:guce:consent-host"/>
<link crossorigin="anonymous" href="//s.yimg.com" rel="preconnect"/><link href="//geo.yahoo.com" rel="preconnect"/><link href="//query1.finance.yahoo.com" rel="preconnect"/><link href="//query2.finance.yahoo.com" rel="preconnect"/><link href="//consent.cmp.oath.com" rel="preconnect"/><link as="worker" href="/__rapidworker-1.2.js" rel="preload"/><link href="https://s.yimg.com/uc/finance/webcore/js/_staticFinProtobuf.4b1559b8e4645fd93a12.js" rel="preconnect"/><link href="https://cdn.jsdelivr.net/npm/protobufjs@7.1.2/dist/minimal/protobuf.min.js" rel="preconnect"/><script>window.finNeoPageStart = Date.now();</script><script>(function(){if(!window.YAHOO){window.YAHOO={}}; window.YAHOO.context={"consent":{"allowContentPersonalization":false,"allowCrossDeviceMapping":false,"al

In [73]:
articles = soup.find("div", {"class": "topic-stories"}).find_all('a')
articles

[<a aria-label="Bank of Canada urged to 'take a breather' and hold rates as Trump rattles trade" class="subtle-link fin-size-small thumb yf-1xqzjha" data-ylk="elm:img;elmt:link;itc:0;ct:story;slk:Bank%20of%20Canada%20urged%20to%20'take%20a%20breather'%20and%20hold%20rates%20as%20Trump%20rattles%20trade;sec:topic-hero;g:63a2c497-dc5c-4cb4-ab91-d35dd95ba095" href="https://ca.finance.yahoo.com/news/bank-of-canada-urged-to-take-a-breather-and-hold-rates-as-trump-rattles-trade-171013098.html" title="Bank of Canada urged to 'take a breather' and hold rates as Trump rattles trade"><div style="display: contents; --border-radius:var(--border-radius-m); --aspect-ratio:16/9;"> <img alt="Bank of Canada urged to 'take a breather' and hold rates as Trump rattles trade" class="tw-bg-opacity-25 yf-13q9uv1" fetchpriority="auto" loading="lazy" src="https://s.yimg.com/uu/api/res/1.2/1G0sVJS_0P51w9yerZJylg--~B/Zmk9c3RyaW07aD0xMjY7cT04MDt3PTE2ODthcHBpZD15dGFjaHlvbg--/https://s.yimg.com/os/creatr-uploaded-i

In [74]:
# article_links = ["https://ca.finance.yahoo.com" + article.get("href") for article in articles]
article_links = [article.get("href") for article in articles]
article_links

['https://ca.finance.yahoo.com/news/bank-of-canada-urged-to-take-a-breather-and-hold-rates-as-trump-rattles-trade-171013098.html',
 'https://ca.finance.yahoo.com/news/bank-of-canada-urged-to-take-a-breather-and-hold-rates-as-trump-rattles-trade-171013098.html',
 'https://ca.finance.yahoo.com/news/i-dont-see-25-sticking-bmo-chief-investment-officer-on-trumps-tariff-plan-174934976.html',
 'https://ca.finance.yahoo.com/news/i-dont-see-25-sticking-bmo-chief-investment-officer-on-trumps-tariff-plan-174934976.html',
 'https://ca.finance.yahoo.com/news/amid-the-housing-crisis-canadians-see-a-big-election-issue-with-no-good-leaders-150017433.html',
 'https://ca.finance.yahoo.com/news/amid-the-housing-crisis-canadians-see-a-big-election-issue-with-no-good-leaders-150017433.html',
 'https://ca.finance.yahoo.com/news/canadas-inflation-rate-edged-down-to-18-in-december-as-gst-break-kicked-in-133531065.html',
 'https://ca.finance.yahoo.com/news/canadas-inflation-rate-edged-down-to-18-in-december-as

In [93]:
scraped_text = dict()
next_articles = []

for i, article_link in enumerate(article_links):
    
    print(i, end='\r')
    
    soup = BeautifulSoup(requests.get(article_link).text)
    title = soup.title.string
    content = soup.find("div", {"class": "body-wrap"}).find_all("p")
    content_text = '\n'.join([p.text for p in content])
    
    next_text = soup.find("div", {"class": "body-wrap"}).find("a")
    next_articles.append(next_text.get("href"))
    scraped_text[title] = content_text

In [94]:
scraped_text

{"Bank of Canada urged to 'take a breather' and hold rates as Trump rattles trade": 'The Bank of Canada should hold its benchmark interest rate steady at 3.25 per cent at its upcoming January meeting, giving policymakers time to digest U.S. President Donald Trump’s vision for Canada-U.S. trade policy.\nThat’s the view from Scotiabank’s vice-president and head of capital markets economics, and Chartered Professional Accountants of Canada.\nTrump has not backed away from Feb. 1 as a start date for blanket 25 per cent tariffs on imports from Canada and Mexico. At the same time, he has ordered federal agencies to study trade policies and trade deficits, and begin consultations on the U.S.-Mexico-Canada Agreement, which is due for review next year.\nOn Tuesday, Canada’s Industry Minister François-Philippe Champagne said the government must “move from a defensive position to offensive.” Prime Minister Justin Trudeau says he is in favour of imposing "matching" retaliatory tariffs on the Unite

In [95]:
next_articles

['https://ca.finance.yahoo.com/news/canada-focus-larger-vision-talks-172956655.html',
 'https://ca.finance.yahoo.com/news/canada-focus-larger-vision-talks-172956655.html',
 'https://ca.finance.yahoo.com/news/trump-plans-enact-25-tariffs-005722671.html',
 'https://ca.finance.yahoo.com/news/trump-plans-enact-25-tariffs-005722671.html',
 'https://ca.finance.yahoo.com/news/canadas-housing-affordability-improves-again-but-homes-still-not-affordable-132955271.html',
 'https://ca.finance.yahoo.com/news/canadas-housing-affordability-improves-again-but-homes-still-not-affordable-132955271.html',
 'https://ca.finance.yahoo.com/news/canadas-annual-inflation-rate-edged-down-to-19-per-cent-in-november-reinforcing-gradual-path-for-boc-150339645.html',
 'https://ca.finance.yahoo.com/news/canadas-annual-inflation-rate-edged-down-to-19-per-cent-in-november-reinforcing-gradual-path-for-boc-150339645.html']